In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, input_dim, output_dim, num_experts, k=2):
        super().__init__()
        self.k = k
        self.experts = nn.ModuleList([
            Expert(input_dim, input_dim * 2, output_dim) 
            for _ in range(num_experts)
        ])
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        gate_logits = self.gate(x)
        weights, indices = torch.topk(gate_logits, self.k, dim=-1)
        weights = F.softmax(weights, dim=-1)

        batch_size = x.size(0)
        output_dim = self.experts[0].net[-1].out_features
        combined_output = torch.zeros(batch_size, output_dim, device=x.device)

        # Vectorized expert application for better performance
        for i in range(self.k):
            exp_indices = indices[:, i]
            exp_weights = weights[:, i].unsqueeze(1)
            
            for idx in range(len(self.experts)):
                mask = (exp_indices == idx)
                if mask.any():
                    expert_out = self.experts[idx](x[mask])
                    combined_output[mask] += exp_weights[mask] * expert_out

        return combined_output, gate_logits

class RecommenderMoE(nn.Module, BaseEstimator, TransformerMixin):
    def __init__(
            self,
            input_dim,
            output_dim=3,
            num_experts=8,
            k=2,
            task_weights=[1.0, 1.0, 1.0],
            epochs=10,
            lr=0.001,
            batch_size=32,
            balancing_coef=0.01,
            device="cpu",
            seed=42
        ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_experts = num_experts
        self.k = k
        self.task_weights = task_weights
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.balancing_coef = balancing_coef
        self.device = torch.device(device)
        self.seed = seed

        self.moe_layer = MoELayer(input_dim, output_dim, num_experts, k)
        self.to(self.device)

    def forward(self, x):
        return self.moe_layer(x)

    def _set_seed(self, seed):
        torch.manual_seed(seed)
        np.random.seed(seed)

    def fit(self, X, y):
        self._set_seed(self.seed)
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).to(self.device)
        t_weights = torch.tensor(self.task_weights, dtype=torch.float32).to(self.device)
        
        loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=self.batch_size, shuffle=True)
        optimizer = optim.Adam(self.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.train()
        for epoch in range(self.epochs):
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                preds, gate_logits = self.forward(batch_x)
                
                # Pareto-weighted Task Loss
                # Calculates individual losses for Likes, Watch Time, and CTR
                diffs = (preds - batch_y)**2
                task_loss = (diffs * t_weights).mean()

                # Load Balancing Loss
                importance = torch.mean(F.softmax(gate_logits, dim=-1), dim=0)
                load_loss = torch.std(importance) / (torch.mean(importance) + 1e-6)

                total_loss = task_loss + (self.balancing_coef * load_loss)
                total_loss.backward()
                optimizer.step()
        return self

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
            preds, _ = self.forward(X_tensor)
            return preds.cpu().numpy()

In [ ]:
import numpy as np
# 1. Prepare your targets (y)
# Column 0: Like (0 or 1)
# Column 1: Watch time (e.g., minutes)
# Column 2: Click-through (0 or 1)
num_samples = 1000
input_features = 50
output_labels = 3

X_train = np.random.rand(num_samples, input_features).astype(np.float32)
y_train = np.random.rand(num_samples, output_labels).astype(np.float32)

# 2. Initialize with output_dim=3
recommender = RecommenderMoE(
    input_dim=50,
    output_dim=3, # Predicting 3 metrics at once
    num_experts=8,
    k=2
)

# 3. Train
recommender.fit(X_train, y_train)

# 4. Predict
new_user_data = np.random.rand(1, input_features).astype(np.float32)
predictions = recommender.predict(new_user_data)
likes, watch_time, ctr = predictions[0]